In [2]:
# Импорт библиотек
import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torchvision import transforms
from PIL import Image
from pathlib import Path
from pycocotools.coco import COCO
from tqdm import tqdm
import random

In [3]:
# Проверка доступности GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")
print(f"Название GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


Используемое устройство: cuda
Название GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [4]:

# Пути к данным
BASE_DIR = Path("archive/coco2017")
TRAIN_IMAGE_DIR = BASE_DIR / "filtered_train2017"
VAL_IMAGE_DIR = BASE_DIR / "filtered_val2017"
TRAIN_ANNOTATION = BASE_DIR / "annotations/filtered_train_annotations.json"
VAL_ANNOTATION = BASE_DIR / "annotations/filtered_val_annotations.json"

# Проверка существования путей
print("\nПроверка путей:")
print(f"Тренировочные изображения: {TRAIN_IMAGE_DIR.exists()}")
print(f"Валидационные изображения: {VAL_IMAGE_DIR.exists()}")
print(f"Тренировочные аннотации: {TRAIN_ANNOTATION.exists()}")
print(f"Валидационные аннотации: {VAL_ANNOTATION.exists()}")



Проверка путей:
Тренировочные изображения: True
Валидационные изображения: True
Тренировочные аннотации: True
Валидационные аннотации: True


In [5]:
# Загрузка аннотаций
train_coco = COCO(TRAIN_ANNOTATION)
val_coco = COCO(VAL_ANNOTATION)

# Вывод информации о датасете
print("\nИнформация о датасете:")
print(f"Тренировочные изображения: {len(train_coco.imgs)}")
print(f"Тренировочные аннотации: {len(train_coco.anns)}")
print(f"Валидационные изображения: {len(val_coco.imgs)}")
print(f"Валидационные аннотации: {len(val_coco.anns)}")
print(f"Категории: {[cat['name'] for cat in train_coco.cats.values()]}")

loading annotations into memory...
Done (t=0.68s)
creating index...
index created!
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!

Информация о датасете:
Тренировочные изображения: 15558
Тренировочные аннотации: 52905
Валидационные изображения: 673
Валидационные аннотации: 2267
Категории: ['bottle', 'wine glass', 'cup']


In [6]:
# Блок 3: Параметры обработки изображений и трансформации
IMG_SIZE = (256, 256)  # Размер изображения для сети
BATCH_SIZE = 16        # Размер батча

# Проверка доступности GPU
print(f"Доступно GPU: {torch.cuda.is_available()}")
print(f"Количество GPU: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Имя GPU: {torch.cuda.get_device_name(0)}")

# Безопасные трансформации (без аугментаций)
try:
    # Трансформации только для изображений
    image_transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),  # Ресайз до нужного размера
        transforms.ToTensor(),         # Конвертация в тензор [0,1]
        transforms.Normalize(          # Нормализация по каналам
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )
    ])
    print("Трансформации успешно созданы")
except Exception as e:
    print(f"Ошибка при создании трансформаций: {str(e)}")
    # Простые трансформации на случай ошибки
    image_transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor()
    ])

Доступно GPU: True
Количество GPU: 1
Имя GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Трансформации успешно созданы


In [8]:
# Блок 4: Кастомный Dataset класс для COCO с обработкой ошибок
class COCOSegmentationDataset(Dataset):
    def __init__(self, coco, img_dir, transform=None):
        self.coco = coco          # COCO API объект
        self.img_dir = img_dir    # Путь к изображениям
        self.transform = transform  # Трансформации для изображений
        self.ids = list(coco.imgs.keys())  # ID всех изображений
        
    def __len__(self):
        return len(self.ids)  # Количество изображений
    
    def __getitem__(self, idx):
        try:
            # Получаем ID изображения
            img_id = self.ids[idx]
            
            # Загружаем метаданные изображения
            img_info = self.coco.loadImgs(img_id)[0]
            img_width, img_height = img_info['width'], img_info['height']
            
            # Формируем путь и загружаем изображение
            img_path = self.img_dir / img_info['file_name']
            if not img_path.exists():
                raise FileNotFoundError(f"Файл {img_path} не найден")
                
            img = Image.open(img_path).convert('RGB')
            
            # Создаем пустую маску
            mask = np.zeros((img_height, img_width), dtype=np.uint8)
            
            # Получаем все аннотации для изображения
            ann_ids = self.coco.getAnnIds(imgIds=img_id)
            anns = self.coco.loadAnns(ann_ids)
            
            # Добавляем объекты на маску
            for ann in anns:
                obj_mask = self.coco.annToMask(ann)
                mask = np.maximum(mask, obj_mask * 255)
            
            # Конвертируем маску в PIL Image
            mask = Image.fromarray(mask)
            
            # Применяем трансформации к изображению
            if self.transform:
                img = self.transform(img)
                
            # Обработка маски (ТОЛЬКО ресайз)
            mask = mask.resize(IMG_SIZE, Image.NEAREST)  # Безопасный ресайз
            mask = transforms.ToTensor()(mask)
            mask = (mask > 0).float()  # Бинаризация
            
            return img, mask
        
        except Exception as e:
            print(f"Ошибка при обработке изображения {idx}: {str(e)}")
            # Возвращаем пустышку вместо падения
            dummy_img = torch.zeros(3, IMG_SIZE[0], IMG_SIZE[1])
            dummy_mask = torch.zeros(1, IMG_SIZE[0], IMG_SIZE[1])
            return dummy_img, dummy_mask

In [9]:
# Блок 5: Создание датасетов с проверкой
print("\nПроверка путей к изображениям:")
print(f"TRAIN_IMAGE_DIR exists: {TRAIN_IMAGE_DIR.exists()}")
print(f"VAL_IMAGE_DIR exists: {VAL_IMAGE_DIR.exists()}")

if not TRAIN_IMAGE_DIR.exists():
    print(f"Ошибка: {TRAIN_IMAGE_DIR} не существует!")
    
if not VAL_IMAGE_DIR.exists():
    print(f"Ошибка: {VAL_IMAGE_DIR} не существует!")

train_dataset = COCOSegmentationDataset(
    coco=train_coco,
    img_dir=TRAIN_IMAGE_DIR,
    transform=image_transform
)

val_dataset = COCOSegmentationDataset(
    coco=val_coco,
    img_dir=VAL_IMAGE_DIR,
    transform=image_transform
)

print(f"Размер тренировочного датасета: {len(train_dataset)}")
print(f"Размер валидационного датасета: {len(val_dataset)}")


Проверка путей к изображениям:
TRAIN_IMAGE_DIR exists: True
VAL_IMAGE_DIR exists: True
Размер тренировочного датасета: 15558
Размер валидационного датасета: 673


In [10]:
# Блок 6: Создание DataLoader'ов с уменьшенными параметрами
train_loader = DataLoader(
    train_dataset,
    batch_size=min(BATCH_SIZE, 4),  # Уменьшенный размер батча для отладки
    shuffle=True,
    num_workers=0,  # Отключаем multiprocessing для отладки
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=min(BATCH_SIZE, 4),
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [11]:
# Блок 7: Безопасная визуализация одного образца
def safe_visualize_sample(dataset, idx=None):
    """Визуализация одного образца с обработкой ошибок"""
    if idx is None:
        idx = random.randint(0, len(dataset)-1)
    
    try:
        img, mask = dataset[idx]
        
        # Денормализация изображения
        img_denorm = img.clone().permute(1, 2, 0).numpy()
        img_denorm = img_denorm * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_denorm = np.clip(img_denorm, 0, 1)
        
        # Преобразование маски
        mask_np = mask.squeeze().numpy()
        
        # Создаем фигуру
        fig, ax = plt.subplots(1, 2, figsize=(10, 5))
        
        # Отображение изображения
        ax[0].imshow(img_denorm)
        ax[0].set_title(f"Изображение {idx}")
        ax[0].axis('off')
        
        # Отображение маски
        ax[1].imshow(mask_np, cmap='gray')
        ax[1].set_title(f"Маска {idx}")
        ax[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Ошибка визуализации образца {idx}: {str(e)}")

# Визуализация первого образца из тренировочных данных
print("\nПервый образец из тренировочного набора:")
safe_visualize_sample(train_dataset, idx=0)

# Визуализация случайного образца из валидационных данных
print("\nСлучайный образец из валидационного набора:")
safe_visualize_sample(val_dataset)


Первый образец из тренировочного набора:


: 